In [1]:
import pandas as pd
import numpy as np
from nltk.tokenize import word_tokenize
from gensim.models import Word2Vec

In [9]:
df = pd.read_csv('data/lemmatized.csv')
tfidf_df = pd.read_csv('data/tfidf_dataset.csv', index_col=0)
w2v_model = Word2Vec.load("word2vec.model")
dim = w2v_model.vector_size

In [10]:
tokenized_texts = [word_tokenize(str(t).lower()) for t in df['text']]

In [11]:
def get_sentence_vector(words, model, tfidf_row, dim):
    vecs = []
    weights = []
    
    for word in words:
        if word in model.wv.key_to_index and word in tfidf_row:
            vecs.append(model.wv[word] * tfidf_row[word])
            weights.append(tfidf_row[word])
    
    if len(vecs) == 0:
        return np.zeros(dim)
    
    return np.average(vecs, axis=0, weights=weights)

In [12]:
vectors = np.vstack([
    get_sentence_vector(words, w2v_model, tfidf_df.iloc[i], dim)
    for i, words in enumerate(tokenized_texts)
])

In [13]:
columns = [f's2v_{i}' for i in range(dim)]
s2v_df = pd.DataFrame(vectors, columns=columns)
df_s2v = pd.concat([df.reset_index(drop=True), s2v_df.reset_index(drop=True)], axis=1)

In [14]:
df_s2v.to_csv('data/sentence2vec_dataset.csv', index=False)
df_s2v.head()

,Unnamed: 0,title,text,label,s2v_0,s2v_1,s2v_2,s2v_3,s2v_4,s2v_5,...,s2v_90,s2v_91,s2v_92,s2v_93,s2v_94,s2v_95,s2v_96,s2v_97,s2v_98,s2v_99
0,0,LAW ENFORCEMENT HIGH ALERT Following Threats C...,comment expected Barack Obama Members FYF911 F...,1,-0.036621,0.021288,0.003388,0.007990,-0.019836,-0.087577,...,0.033387,0.001450,-0.025691,-0.016768,0.050253,0.001316,0.040150,-0.021223,0.013774,-0.033349
1,1,NaN,post vote Hillary already,1,-0.076511,0.197400,0.189406,-0.159344,0.066331,-0.335332,...,0.181832,0.175117,-0.108259,-0.184448,0.244396,-0.039861,0.021803,-0.056795,-0.178185,-0.056384
2,2,UNBELIEVABLE OBAMAS ATTORNEY GENERAL SAYS CHAR...,demonstrator gathered last night exercising co...,1,-0.019873,0.052981,0.005044,0.026973,-0.027834,-0.157035,...,0.117663,-0.006514,-0.089740,0.020153,0.150888,-0.001570,0.079693,-0.105059,-0.004903,-0.072006
3,3,Bobby Jindal raised Hindu us story Christian c...,dozen politically active pastor came private d...,0,-0.014440,0.004744,0.017398,0.027681,-0.019082,-0.102524,...,0.022037,0.016291,-0.001010,0.015831,0.036998,-0.029930,0.027266,-0.015379,-0.013087,-0.014944
4,4,SATAN 2 Russia unvelis image terrifying new SU...,RS28 Sarmat missile dubbed Satan 2 replace SS1...,1,-0.040809,-0.029174,0.041273,-0.057825,-0.014951,-0.100982,...,0.067549,-0.027068,0.030822,-0.063464,0.096822,-0.097038,0.136594,-0.051239,0.050671,-0.095231
